# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a walkthrough for loading and exploring the FAIR⁲ dataset using the `mlcroissant` library. All datasets, record sets, fields, and columns are referenced by their Croissant `@id`.

### Dataset Source
The dataset schema is provided at: [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)


In [ ]:
# Ensure mlcroissant is installed
!pip install --quiet mlcroissant

## 1. Data Loading
Load dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset object
ds = mlc.Dataset(croissant_url)
metadata = ds.metadata

print(f"Dataset: {metadata.name}\nDescription: {metadata.description}\n")

## 2. Data Overview
Let's review the available record sets and their fields. We will reference all entities by their Croissant `@id`.

In [ ]:
# List all record sets with their @id, name, and fields
record_sets = metadata.record_sets
print(f"Number of record sets: {len(record_sets)}\n")
rs_info = {}
for rs in record_sets:
    print(f"RecordSet @id: {rs.id}\n  name:   {rs.name}\n  description: {rs.description}")
    print("  Fields:")
    rs_info[rs.id] = []
    for field in rs.fields:
        rs_info[rs.id].append(field.id)
        print(f"    - {field.id}" + (f" (name: {field.name})" if field.name else ""))
    print("")
# Select a record set for demo, picking the first
main_record_set_id = record_sets[0].id if record_sets else None

## 3. Data Extraction
Load data from a specific record set into a DataFrame. Choose a record set and show its records as a pandas DataFrame for analysis.

In [ ]:
# List of all record set @ids
record_set_ids = [rs.id for rs in metadata.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    print(f"Loading records from RecordSet: {record_set_id}")
    records = list(ds.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"  Columns for {record_set_id}:", df.columns.tolist())
    print(df.head(2))
    print()
# Select main_record_set_id for further EDA
print(f"Main record set chosen for EDA: {main_record_set_id}")
main_df = dataframes[main_record_set_id]

## 4. Exploratory Data Analysis (EDA)
We perform EDA by filtering and transforming numeric fields, and grouping data if applicable. All fields are referenced by their `@id`.


In [ ]:
# Choose a numeric field (using @id)--show list of possible numeric fields
numeric_field_id = None
print("Available columns for EDA:", main_df.columns.tolist())

for col in main_df.columns:
    # Try to infer numeric fields
    if pd.api.types.is_numeric_dtype(main_df[col]):
        numeric_field_id = col
        break

if numeric_field_id is None:
    # Try to force-convert possible candidates
    for col in main_df.columns:
        try:
            main_df[col] = pd.to_numeric(main_df[col])
            if pd.api.types.is_numeric_dtype(main_df[col]):
                numeric_field_id = col
                break
        except Exception:
            pass

print(f"Numeric field selected: {numeric_field_id}")

# EDA: filter and normalize
if numeric_field_id is not None:
    threshold = main_df[numeric_field_id].mean() if main_df[numeric_field_id].notna().any() else 0
    filtered_df = main_df[main_df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    print(filtered_df.head())

    # Normalization
    filt_norm_col = f"{numeric_field_id}_normalized"
    filtered_df[filt_norm_col] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
        filtered_df[numeric_field_id].std()
    )
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, filt_norm_col]].head())

    # Attempt group by a categorical field (e.g., the 2nd column)
    group_field = None
    for col in main_df.columns:
        if col != numeric_field_id and main_df[col].nunique() < 10:
            group_field = col
            break

    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().to_frame()
        print(f"\nGrouped and averaged by {group_field}:")
        print(grouped_df.head())
else:
    print("No numeric field found in this RecordSet for EDA.")

## 5. Visualization
Visualize the distribution of the numeric field or the relationship with a group field, if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id is not None:
    # Histogram
    plt.figure(figsize=(7,4))
    sns.histplot(main_df[numeric_field_id].dropna(), bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # If group_field exists, show boxplot
    if group_field:
        plt.figure(figsize=(7,4))
        sns.boxplot(data=main_df, x=group_field, y=numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field}")
        plt.show()
else:
    print("No numeric field to visualize.")

## 6. Conclusion
In this notebook, we loaded the FAIR⁲ dataset using `mlcroissant`, explored available record sets and fields using their Croissant `@id`s, extracted a main table, and performed initial data cleaning, normalization, and simple visualizations. This process enables structured, reproducible analysis for mass spectrometry-based proteomics datasets.

**Next steps:** You may proceed to more advanced processing, machine learning, or deeper domain analysis using these extracted tables.